# AIC 2025 — GPU TransNet V2 Shot Segmentation & Smart Merge

> **Recommended Kaggle Accelerator**: **GPU T4 x 2** (Settings $\to$ Accelerator $\to$ GPU T4 x 2).

Fast GPU batch video segmentation using **TransNet V2** (3D-CNN on CUDA) and **`smart_merge_shots`**.

- **Runtime**: ~5-7s per video (~2 min for 29 videos, ~25 min for 300 videos on T4x2 GPU).
- **Smart Merge**: Iterative greedy shortest-neighbor merge ($T_{min} = 10.0$s) matching `making_shot`.
- **Output**: Zipped `shot_boundaries.zip` containing `{video_id}.json` files ready for ReCap.

In [ ]:
# 1. Install dependencies
!pip install -q transnetv2-pytorch tqdm opencv-python ffmpeg-python

In [ ]:
import os
import json
import shutil
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transnetv2_pytorch import TransNetV2

# 1. Helper Functions
def to_seconds(val) -> float:
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        if ":" in val:
            parts = val.split(":")
            if len(parts) == 3:
                return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
            elif len(parts) == 2:
                return float(parts[0]) * 60 + float(parts[1])
        return float(val)
    return float(val)

def smart_merge_shots(shots: list, min_duration: float = 10.0) -> list:
    if not shots:
        return []
    cleaned_shots = []
    for s in shots:
        st = to_seconds(s["start_time"])
        et = to_seconds(s["end_time"])
        cleaned_shots.append({
            "start_time": st,
            "end_time": et,
            "duration": et - st,
            "start_frame": int(s.get("start_frame", 0)),
            "end_frame": int(s.get("end_frame", 0))
        })
    while True:
        if len(cleaned_shots) <= 1:
            break
        min_shot_idx = min(range(len(cleaned_shots)), key=lambda i: cleaned_shots[i]["duration"])
        min_shot = cleaned_shots[min_shot_idx]
        if min_shot["duration"] >= min_duration:
            break
        left_idx = min_shot_idx - 1 if min_shot_idx > 0 else None
        right_idx = min_shot_idx + 1 if min_shot_idx < len(cleaned_shots) - 1 else None
        if left_idx is not None and right_idx is not None:
            target_idx = left_idx if cleaned_shots[left_idx]["duration"] <= cleaned_shots[right_idx]["duration"] else right_idx
        elif left_idx is not None:
            target_idx = left_idx
        else:
            target_idx = right_idx
        if target_idx < min_shot_idx:
            cleaned_shots[target_idx]["end_time"] = min_shot["end_time"]
            cleaned_shots[target_idx]["end_frame"] = min_shot["end_frame"]
            cleaned_shots[target_idx]["duration"] = cleaned_shots[target_idx]["end_time"] - cleaned_shots[target_idx]["start_time"]
            cleaned_shots.pop(min_shot_idx)
        else:
            cleaned_shots[target_idx]["start_time"] = min_shot["start_time"]
            cleaned_shots[target_idx]["start_frame"] = min_shot["start_frame"]
            cleaned_shots[target_idx]["duration"] = cleaned_shots[target_idx]["end_time"] - cleaned_shots[target_idx]["start_time"]
            cleaned_shots.pop(min_shot_idx)
    return cleaned_shots

# 2. Path Setup
OUTPUT_DIR = Path("/kaggle/working/shot_boundaries")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Discover Videos Across /kaggle/input
all_videos = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(".mp4"):
            all_videos.append(Path(root) / f)
all_videos = sorted(all_videos)

# Set START_VIDEO_ID = None to process all, or e.g. 'L26_V128' to resume from a specific ID
START_VIDEO_ID = None
if START_VIDEO_ID:
    video_paths = [p for p in all_videos if p.stem >= START_VIDEO_ID]
else:
    video_paths = all_videos

print(f"Total videos discovered: {len(all_videos)}")
print(f"Total videos to process: {len(video_paths)}")
if video_paths:
    print("Sample files:", [p.name for p in video_paths[:3]])

# 4. Load TransNetV2 Model on CUDA GPU (T4 x 2)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

try:
    print("Initializing TransNetV2 on GPU...")
    model = TransNetV2(device=device)
except Exception as e:
    print(f"CUDA initialization fallback ({e}). Using CPU...")
    model = TransNetV2(device="cpu")

# 5. Process Videos
for vp in tqdm(video_paths, desc="TransNet GPU Shots"):
    video_name = vp.stem
    out_json = OUTPUT_DIR / f"{video_name}.json"
    if out_json.exists():
        continue
    try:
        raw_scenes = model.detect_scenes(vp)
        raw_shots = []
        for s in raw_scenes:
            st = to_seconds(s["start_time"])
            et = to_seconds(s["end_time"])
            raw_shots.append({
                "start_time": st,
                "end_time": et,
                "start_frame": int(s.get("start_frame", 0)),
                "end_frame": int(s.get("end_frame", 0)),
                "duration": et - st
            })
        final_shots = smart_merge_shots(raw_shots, min_duration=10.0)
        output_payload = []
        for shot_id, s in enumerate(final_shots):
            st = round(s["start_time"], 2)
            et = round(s["end_time"], 2)
            mid_pts = round((st + et) / 2.0, 3)
            mid_idx = int((s["start_frame"] + s["end_frame"]) / 2)
            output_payload.append({
                "shot_id": shot_id + 1,
                "start_time": st,
                "end_time": et,
                "duration": round(s["duration"], 2),
                "n_keyframes": 1,
                "keyframes": [{
                    "n": 1,
                    "pts_time": mid_pts,
                    "fps": 25.0,
                    "frame_idx": mid_idx,
                    "image": f"{mid_idx:06d}.jpg"
                }]
            })
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(output_payload, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"Error {video_name}: {e}")

# 6. Zip Output for 1-Click Download
generated = list(OUTPUT_DIR.glob("*.json"))
print(f"\nDone! Generated {len(generated)} shot boundary files.")
shutil.make_archive("/kaggle/working/shot_boundaries", "zip", OUTPUT_DIR)
print("Download ready at: /kaggle/working/shot_boundaries.zip")